In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
train_dir = '/kaggle/input/competitions/2026task2/fer_data/fer_data/train'
test_dir  = '/kaggle/input/competitions/2026task2/fer_data/fer_data/test'
classes = sorted(os.listdir(train_dir))
print("表情类别:", classes) 

表情类别: ['Angry', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']


In [3]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
IMG_SIZE = (48, 48)
BATCH = 64
# 训练集做增强，防止过拟合
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1,
    validation_split=0.2   # 从训练集划出20%做验证
)
train_data = train_gen.flow_from_directory(
    train_dir, target_size=IMG_SIZE, color_mode='grayscale',
    batch_size=BATCH, class_mode='categorical', subset='training', shuffle=True
)
val_data = train_gen.flow_from_directory(
    train_dir, target_size=IMG_SIZE, color_mode='grayscale',
    batch_size=BATCH, class_mode='categorical', subset='validation', shuffle=False
)
# 测试集只做归一化
test_gen = ImageDataGenerator(rescale=1./255)
test_data = test_gen.flow_from_directory(
    test_dir, target_size=IMG_SIZE, color_mode='grayscale',
    batch_size=1, class_mode=None, shuffle=False
)
print("类别索引映射:", train_data.class_indices)

Found 22623 images belonging to 6 classes.
Found 5652 images belonging to 6 classes.
Found 0 images belonging to 0 classes.
类别索引映射: {'Angry': 0, 'Fear': 1, 'Happy': 2, 'Neutral': 3, 'Sad': 4, 'Surprise': 5}


In [4]:
from tensorflow.keras import layers, models

def build_model(num_classes=6):
    model = models.Sequential([
        layers.Input(shape=(48, 48, 1)),
        layers.Conv2D(64, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(64, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(2),
        layers.Dropout(0.25),

        layers.Conv2D(128, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(128, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(2),
        layers.Dropout(0.25),

        layers.Conv2D(256, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(256, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(2),
        layers.Dropout(0.4),

        layers.Flatten(),
        layers.Dense(512, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

model = build_model()
model.summary()

I0000 00:00:1791975603.786082      49 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1791975603.789505      49 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 48, 48, 64)     │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 48, 48, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 48, 48, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 24, 24, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 24, 24, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 24, 24, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 24, 24, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 24, 24, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 12, 12, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 12, 12, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 12, 12, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 6, 6, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 6, 6, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 9216)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │     4,719,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │         3,078 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,870,278 (22.39 MB)

 Trainable params: 5,868,358 (22.39 MB)

 Non-trainable params: 1,920 (7.50 KB)

In [5]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True,
                                     monitor='val_accuracy'),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3,
                                         monitor='val_loss'),
    tf.keras.callbacks.ModelCheckpoint('/kaggle/working/best_model.keras',
                                       save_best_only=True, monitor='val_accuracy')
]

history = model.fit(train_data, validation_data=val_data,
                    epochs=60, callbacks=callbacks)

Epoch 1/60
  1/354 ━━━━━━━━━━━━━━━━━━━━ 1:14:56 13s/step - accuracy: 0.1875 - loss: 2.9045

I0000 00:00:1791975631.159723     163 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


354/354 ━━━━━━━━━━━━━━━━━━━━ 225s 602ms/step - accuracy: 0.2491 - loss: 1.9828 - val_accuracy: 0.2420 - val_loss: 1.8158 - learning_rate: 0.0010
Epoch 2/60
354/354 ━━━━━━━━━━━━━━━━━━━━ 39s 109ms/step - accuracy: 0.3480 - loss: 1.6701 - val_accuracy: 0.2785 - val_loss: 1.9166 - learning_rate: 0.0010
Epoch 3/60
354/354 ━━━━━━━━━━━━━━━━━━━━ 38s 108ms/step - accuracy: 0.4379 - loss: 1.4348 - val_accuracy: 0.4885 - val_loss: 1.3256 - learning_rate: 0.0010
Epoch 4/60
354/354 ━━━━━━━━━━━━━━━━━━━━ 36s 102ms/step - accuracy: 0.4899 - loss: 1.3038 - val_accuracy: 0.5016 - val_loss: 1.2804 - learning_rate: 0.0010
Epoch 5/60
354/354 ━━━━━━━━━━━━━━━━━━━━ 37s 104ms/step - accuracy: 0.5125 - loss: 1.2486 - val_accuracy: 0.5168 - val_loss: 1.2486 - learning_rate: 0.0010
Epoch 6/60
354/354 ━━━━━━━━━━━━━━━━━━━━ 40s 112ms/step - accuracy: 0.5326 - loss: 1.2028 - val_accuracy: 0.4630 - val_loss: 1.3606 - learning_rate: 0.0010
Epoch 7/60
354/354 ━━━━━━━━━━━━━━━━━━━━ 41s 116ms/step - accuracy: 0.5498 - loss

In [6]:
from tensorflow.keras.preprocessing.image import load_img, img_to_array
test_files = sorted(os.listdir(test_dir))
print("测试图片数量:", len(test_files))   # 先确认不是 0
pred_labels = []
for f in test_files:
    img = load_img(os.path.join(test_dir, f),
                   target_size=(48, 48), color_mode='grayscale')
    x = img_to_array(img) / 255.0          # 归一化，与训练一致
    x = np.expand_dims(x, axis=0)          # 变成 (1, 48, 48, 1)
    pred = np.argmax(model.predict(x, verbose=0), axis=1)[0]
    pred_labels.append(pred)
# 旧索引 -> 新标签
remap = {0: 0, 1: 1, 2: 2, 3: 5, 4: 3, 5: 4}
pred_labels = [remap[int(p)] for p in pred_labels]
submission = pd.DataFrame({'ID': test_files,'Emotion': pred_labels})
submission.to_csv('/kaggle/working/submission.csv', index=False)

测试图片数量: 7065
